## Topic 1: Overview of Class Builders
1. What It Is

- When writing Python classes that primarily act as passive data containers (holding fields with little or no business logic), we quickly run into the boilerplate problem. Standard Python requires us to repeat attribute names multiple times in `__init__`, and it does not automatically provide useful implementations for `__repr__`, `__eq__`, or hashing.
- Data class builders are **code generators or decorators** in the standard library that automate this work.

2. The Mechanism
- Python provides three main class builders, each operating under a different mechanism:
    - `collections.namedtuple`: A factory function that dynamically builds a new subclass of the built-in tuple at runtime.
    - `typing.NamedTuple`: Leverages a metaclass at import time to parse class-level variable annotations (type hints) to generate a customized subclass of tuple.
    - `@dataclasses.dataclass`: A class decorator that reads class variable type annotations at import time, programmatically generates standard dunder methods (like `__init__`, `__repr__`, `__eq__`), and injects them directly into a standard user-defined class inheriting from object.

3. Gotchas & Edge Cases (The Non-Obvious)

- The Tuple Inheritance Trap: Because `namedtuple` and `NamedTuple` produce subclasses of `tuple` which is the real base class not `NamedTuple`, their instances are **strictly immutable**. However, this also means they inherit all of tuple's behaviors—including iteration (thus unpacking works too!), slicing, and index-based comparisons. This means a `NamedTuple` instance representing a coordinate pair can be compared directly to a raw 2-tuple (see line 18 in the next cell), which might lead to silent logical bugs if you wanted your record type to be distinct.
- An instance of `dataclass` with two fields can not do equality comparison with a standard tuple as they're of different types! See line 21 on the next cell! Unpacking doesn't work with it either unless `__iter__` was explicitly implemented since by default it's not.
- No Runtime Type Enforcement: Although `NamedTuple` and `@dataclass` require type annotations to discover fields, type hints have absolutely **no impact** on the runtime behavior of Python programs. The constructors generated by these builders will happily accept invalid types (e.g., passing a string where a float was declared) without raising errors or warnings at runtime

In [32]:
# Topic 1 Challenge: Predict the Output
from typing import NamedTuple
from dataclasses import dataclass

class CardNT(NamedTuple):
    rank: str
    suit: str

@dataclass
class CardDC:
    rank: str
    suit: str

c1 = CardNT('A', 'Spades')
c2 = CardDC('A', 'Spades')

# 1. What will this statement print?
print(c1 == ('A', 'Spades')) # True

# 2. What will this statement print?
print(c2 == ('A', 'Spades')) #NOTE: False

# 3. If we execute the following two mutation lines:
c1.rank = 'K' # failed because NamedTuple instances are immutable.
#    c2.rank = 'K' 
#    Which of these assignments will fail, and what specific exception type will be raised?
# will raise AttributeError: can't set attribute

True
False


AttributeError: can't set attribute

In [ ]:
r1, s1 = c1
r2, s2 = c2 # unpacking doesn't work

TypeError: cannot unpack non-iterable CardDC object

## Topic 2: Classic Named Tuples & Fields
1. What It Is
- A classic named tuple (`collections.namedtuple`) is a **factory function** that programmatically builds a custom subclass of tuple enhanced with named fields, a class name, and an informative string representation.
2. The Mechanism: Behind the Factory
- Zero Memory Overhead: Because it subclasses the built-in `tuple`, each instance of a named tuple consumes exactly the same amount of memory as a raw tuple. The field names are stored within the **class dictionary**, not within the individual instances themselves.
- Helper Methods: It automatically attaches specialized class and instance attributes prefixed with a single underscore to avoid name collisions with user-defined fields:
    - `_fields`: A tuple containing the string names of the fields.
    - `_make(iterable)`: A **class** method that constructs a new instance from an iterable (bypassing the need to unpack with * manually).
    - `_asdict()`: An instance method that returns a standard dict mapping field names to their values.
3. Gotchas & Edge Cases (The Non-Obvious)
- Rightmost Defaults Limitation: Since Python 3.7, you can specify default values using the defaults parameter (an iterable). However, these defaults are strictly mapped to the **rightmost** fields of the named tuple first. Just like standard Python positional function arguments, you cannot have a field without a default value defined to the right of a field that has one.
- The Method Injection Hack: Because a classic named tuple is generated by a factory function and not a standard class statement, you cannot write custom methods inside its body. If you need a method, you must define a standard function elsewhere and explicitly assign it as a class attribute after the named tuple class has been generated.

## collections.namedtuple
- It's a factory that builds subclasses of tuple enhanced with field names, class name and an information `__repr__`.

In [74]:
from collections import namedtuple

City = namedtuple('City', 'name country population coordinates', defaults=[309, (10.00, 20.00)])
dalian = City('dalian', 'China')
beijing = City('Beijing', 'China', 204, (23.54, 58.28))
beijing,dalian

(City(name='Beijing', country='China', population=204, coordinates=(23.54, 58.28)),
 City(name='dalian', country='China', population=309, coordinates=(10.0, 20.0)))

In [2]:
beijing.coordinates

(23.54, 58.28)

In [3]:
beijing[1]

'China'

## Some useful additional methods on namedtuple instance

In [4]:
beijing._fields # return a tuple of field names


('name', 'country', 'population', 'coordinates')

In [9]:
Coordinate = namedtuple('Coordinate', 'lat lon')
delhi_data = ('Delhi NCR', 'IN', 21.935, Coordinate(28.5, 65.2))
delhi = City._make(delhi_data) # create a city instance from an iterable of data.
delhi2 = City(*delhi_data) # same as City._make(iterable)

In [6]:
dict_data = delhi._asdict() # returns a dict of field name and value; useful for building json data.
dict_data

{'name': 'Delhi NCR',
 'country': 'IN',
 'population': 21.935,
 'coordinates': Coordinate(lat=28.5, lon=65.2)}

In [8]:
import json
json.dumps(dict_data)

'{"name": "Delhi NCR", "country": "IN", "population": 21.935, "coordinates": [28.5, 65.2]}'

In [36]:
# namedtuple with defaults
Coordiante2 = namedtuple('Coordinate2', 'lon lat reference', defaults=['WGS82'])
london = Coordiante2(lon=32.5, lat=85.2)
london

Coordinate2(lon=32.5, lat=85.2, reference='WGS82')

In [12]:
london._field_defaults, london._fields

({'reference': 'WGS82'}, ('lon', 'lat', 'reference'))

In [38]:
print(london)

Log = namedtuple('Log', 'name coordinates')
my_log = Log(name='Admin', coordinates=[10.0, 20.0])
my_log.coordinates.append(30.0)
print(my_log)

Coordinate2(lon=32.5, lat=85.2, reference='WGS82')
Log(name='Admin', coordinates=[10.0, 20.0, 30.0])


## Topic 3: Typed Named Tuples
1. What It Is
- While classic named tuples (`collections.namedtuple`) are built dynamically using a factory function, `typing.NamedTuple` allows you to declare a typed, structured, **immutable** record class using standard Python class syntax and PEP 526 variable type annotations.
2. The Mechanism
- Despite being declared with standard class inheritance syntax:
```
class Coordinate(typing.NamedTuple):
    lat: float
    lon: float
```
`typing.NamedTuple` is not a typical superclass.
- The Metaclass Magic: Under the hood, `typing.NamedTuple` uses a metaclass to intercept class creation at import time. It inspects the `__annotations__` namespace of the class block to discover fields and their declared types.
- Building a Tuple Subclass: It dynamically generates a concrete subclass of the built-in `tuple`.
- Descriptor Getters: The fields you declare are attached to the class as specialized, read-only property-like **descriptors** (e.g., `<_collections._tuplegetter>`). These descriptors map attribute access (like coord.lat) to the corresponding positional index of the underlying tuple.
3. Gotchas & Edge Cases (The Non-Obvious)
- Type Hints Have No Runtime Effect: The type hints are **purely metadata** for static analysis tools like Mypy. Python's runtime completely ignores them. You can pass a string to a field annotated as float without raising any exception.
- **Annotations vs. Class Attributes**: 
    - In standard class statements, variables declared with type hints **but no default values** (like `lat: float`) **do not** actually become class attributes at all; they exist only as entries in __annotations__. 
    - However, any class body variable declared without a type hint (like `species = 'Human'`) is treated as a standard, mutable **class-level** attribute and is not recognized as a tuple field.
- The Shared Mutable Default Trap: Just like function default parameters, if you set a default value of a field to a mutable object (like a list []), that default is evaluated exactly once when the class is defined. All instances initialized **without that argument** will share the exact same list instance in memory, leading to silent data-sharing bugs!

## Typed Named Tuples

In [75]:
from typing import NamedTuple

class Coordinate3(NamedTuple):
    lon: float
    lat: float
    refernce: str = 'WGS82'

london2 = Coordinate3(lon=42.5, lat=84.2)

In [76]:
class User(NamedTuple):
    username: str
    roles: list = []
    species = 'Human'

u1 = User('alice')
u2 = User('bob')

# Action 1: Mutate the default list of roles
u1.roles.append('admin')

# Action 2: Rebind the class attribute
User.species = 'Cyborg'

# 1. What will this print?
print(len(u1)) # 2 because species is not an instance attribute

# 2. What will this print?
print(u2.roles) # ['admin'] as all instances share the default 

# 3. What will this print?
print(u1.species) # Cyborg as it retrieves species from the class not itself.

2
['admin']
Cyborg


## Type Hint 101
- They have no runtime effect at all and are not enforced by Python bytecode compiler and interpreter.
- They're more like the documentation that can be verified by IDEs and type checkers.

In [3]:
trash = Coordinate3(lon='no', lat=None) # this just runs fine as it's not enforced at the runtime!
trash

Coordinate3(lon='no', lat=None, refernce='WGS82')

In [77]:
class PlainClass:
    a = 0
    b: float = 1.1
    c = 'spam'
    d: int

In [78]:
PlainClass.__annotations__

{'b': float, 'd': int}

In [79]:
PlainClass.a
try:
    #NOTE: Python doesn't have undefined!
    PlainClass.d # it's NOT a class attribute because it doesn't bind to a value!
except AttributeError as e:
    print(e)

type object 'PlainClass' has no attribute 'd'


In [24]:
o = PlainClass()
o.b, o.d

AttributeError: 'PlainClass' object has no attribute 'd'

In [14]:
# all attributes on the class is readonly given tuple is immutable!
class DemoNTClass(NamedTuple):
    a: int # part of annotations and instance attribute
    b: float = 1.1
    c = 'spam'



In [19]:
o2 = DemoNTClass(9)
o2.__annotations__

{'a': int, 'b': float}

In [22]:

o2.a, DemoNTClass.a, DemoNTClass.b, DemoNTClass.c

(9,
 _tuplegetter(0, 'Alias for field number 0'),
 _tuplegetter(1, 'Alias for field number 1'),
 'spam')

In [30]:
o2.a = 10

AttributeError: can't set attribute

In [25]:
# now we can do the same with dataclass decrator

from dataclasses import dataclass

@dataclass(frozen=False)
class DemoDataClass:
    a: int
    b: float = 1.1 # 
    c= 'spam'

DemoDataClass.__annotations__ # contains a and b

{'a': int, 'b': float}

In [26]:
# it doesn't have a as a class attribute because a only exists on the instance!
DemoDataClass.a

AttributeError: type object 'DemoDataClass' has no attribute 'a'

In [27]:
ddc = DemoDataClass(9)
ddc.a

9

In [29]:
# it's mutable by default unless we set frozen = True
ddc.a = 100
ddc.a, ddc.b

(100, 1.1)

In [ ]:
# Field Options

# any field with default value must be followed by other fields having default values.

## Topic 4: Data Class Decorators (@dataclass)
1. What It Is
- Introduced in Python 3.7, the ·@dataclasses.dataclass· decorator is a standard library code generator that inspects your class definitions to automatically inject crucial boilerplate methods (such as `__init__`, `__repr__`, and `__eq__`).
2. The Mechanism
- Unlike named tuple builders, `@dataclass` is **not** a class factory and does not modify your class's inheritance hierarchy.
- Decorator Intervention: It is a class decorator applied to a standard user-defined Python class that inherits directly from object.
- Dunder Generation: At import time, the decorator scans the class body's `__annotations__ ` dictionary to identify fields and their types. It then programmatically generates Python bytecode for the constructor and other comparison dunder methods and binds them directly into the class's namespace.
3. Gotchas & Edge Cases (The Non-Obvious)
- The Default Parameter Ordering Trap: The constructor `__init__` generated by `@dataclass` behaves exactly like a standard Python **function**. Because Python does not allow parameters without default values to follow parameters with default values, the moment you declare a field with a default value, all subsequent fields declared below it in the class statement must also provide defaults. Violating this raises a `TypeError` at import time.
- Hashability is Conditionally Disabled: By default, dataclass instances are mutable. Because mutable objects are unsafe to hash, @dataclass automatically sets the class's `__hash__` method to `None`. To make instances hashable, you must set `frozen=True` (and ensure `eq=True` which is the default), which forces @dataclass to generate a valid `__hash__` method.
    - In Python, hashability is about type, not **immutability**, types like list are unhashable by design, meaning even if `frozen=True` then any attribute being of a unhashable type makes the type unhashable.
    - Hashing is lazy NOT eager; so a unhashable type works fine if it's never hashed!
- The Cost of Emulated Immutability: When you set `frozen=True`, Python emulates immutability by generating `__setattr__` and `__delattr__` methods that raise a `FrozenInstanceError` (a subclass of `AttributeError`) on any write attempt. This emulation comes with a **minor execution overhead** during object initialization and field access compared to C-level immutable tuples.
-

In [41]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ImmutablePoint:
    x: int
    y: int = 0

@dataclass
class MutablePoint:
    x: int
    y: int = 0

p1 = ImmutablePoint(1, 2)
p2 = MutablePoint(1, 2)

# Check 1: Verify hashability of the frozen instance
check_1 = hash(p1) is not None

# Check 2: Try to hash the mutable instance
try:
    hash(p2)
    check_2 = "Hashable"
except TypeError:
    check_2 = "Unhashable"

# Check 3: Attempt to mutate the frozen instance
try:
    p1.x = 10
    check_3 = "Success"
except AttributeError:
    check_3 = "AttributeError"

# What will these print?
print(check_1)
print(check_2)
print(check_3)

True
Unhashable
AttributeError


In [43]:
@dataclass(frozen=True)
class Group:
    group_id: int
    members: list

g = Group(24, ['Alice', 'Bob'])

# hash(g) # this fails too!
g.members.append('Tom')
hash(g) # this would definitely fail.

TypeError: unhashable type: 'list'

## Topic 5: Field Options & Customization
1. What It Is
- While specifying simple defaults (e.g., price: float = 0.0) is sufficient for basic dataclasses, we often need more advanced control over individual fields. The `dataclasses.field()` function allows us to customize how each field behaves during constructor initialization, string representation, equality comparisons, and hashing.
2. The Mechanism: Behind the Descriptor
- When you write `field(...)`, you are constructing a Field descriptor object. The `@dataclass` decorator scans the class namespace, identifies these descriptor objects, and reads their configuration parameters.
    - `default`: A static default value.
    - `default_factory`: A zero-argument callable (like list, dict, or a custom function) that is executed dynamically every time a new instance is created.
    - `init`: If `False`, this field is completely omitted from the generated `__init__` signature, preventing callers from setting it directly during instantiation. Default is `True`.
    - `repr`: If `False`, this field is omitted from the autogenerated `__repr__` string. Default is `True`.
    - `compare`: Default is `True`. If `False`, this field is ignored during equality (==) and ordering operations.
    - `hash`: Default is `None`; will only be used when `compare` is `True`.
3. Gotchas & Edge Cases (The Non-Obvious)
- The Exclusivity Rule: You cannot specify both default and default_factory for the same field. If you do, Python raises a ValueError at import time.
- The Mutable Default Shield: If you attempt to assign a mutable literal as a standard default (e.g., `guests: list = []`), `@dataclass` will raise a helpful `ValueError` to protect you. However, this protection only checks `list`, `dict`, and `set`. If you use another mutable type (like a user-defined custom collection), Python will not catch it, and all instances will share the same reference in memory!
- Positional Signature Shifts: Setting `init=False` on a field removes it from the `__init__` constructor parameter list. However, it does not affect the positional order of fields defined after it. The subsequent fields simply "shift left" to fill the positional slot in the constructor signature.


In [53]:
from dataclasses import field

@dataclass(frozen=True)
class Group:
    name: str = field(default='N/A')
    members: list = field(hash=False, default=())

g3 = Group('admin', members=['Alice'])
hash(g3)


@dataclass
class UserSession:
    username: str
    session_id: str = field(repr=False)
    login_attempts: int = field(compare=False, default=0)

s1 = UserSession(username="alice", session_id="SECURE_ID_A", login_attempts=1)
s2 = UserSession(username="alice", session_id="SECURE_ID_B", login_attempts=5)

2585937370012369506

## Topic 6: Post-init Processing (`__post_init__`)
1. What It Is
- By default, the `__init__` constructor generated by `@dataclass` only takes the declared fields, maps them to constructor arguments, and performs direct assignments. If you need to perform validation, compute fields based on other fields, or do any post-processing after those assignments are made, you utilize the `__post_init__` special method.
2. The Mechanism: Inside the Constructor
- When the decorator generates the `__init__` method, it automatically checks if a method named `__post_init__` exists in the class definition. If it does, the decorator appends a call to `self.__post_init__()` as the very last line of the compiled `__init__` method.
- Init-Only Variables (`InitVar`):
    Sometimes you need to pass configuration parameters or database connections to the constructor that are solely needed for setup but should **not** be stored as permanent fields on the instance. Python provides `dataclasses.InitVar` for this purpose. When a field is annotated with `InitVar` (e.g., `db: InitVar[Database]`), the generator includes it in the `__init__` parameter list, but **excludes** it from becoming a standard instance attribute. The constructor passes this variable directly as an argument to `self.__post_init__(db)`. Consequently, if you use `InitVar` in your fields, your `__post_init__` method signature must accept that parameter in the exact order it was declared.
3. Gotchas & Edge Cases (The Non-Obvious)
- The Frozen Dataclass Collision: If you define a dataclass with `frozen=True`, Python emulates immutability by generating `__setattr__` and` __delattr__` methods that raise `FrozenInstanceError` (an AttributeError) on any modification attempts. If you try to write `self.computed_field = x` inside `__post_init__` on a frozen class, it will crash immediately. To modify or set attributes inside a frozen instance's `__post_init__`, you must bypass the class's overridden setter by calling the base class's setter directly:`object.__setattr__(self, 'computed_field', x)`
- Attribute Access Failures: Attempting to retrieve an `InitVar` on an instance after initialization (e.g., `my_obj.db`) will fail and raise an `AttributeError` because it was **never** bound to the object dictionary.
    - NOTE: that the instance retrieval would work if the field has a default value, which would make it part of the class attributes thus retrieving it on an instance still works (see print #3 from cell below!).

In [55]:
from dataclasses import dataclass, InitVar

@dataclass
class Member:
    name: str
    age: int
    multiplier: InitVar[int] = 2
    score: int = 0

    def __post_init__(self, multiplier):
        self.score = self.age * multiplier
        if self.name == "":
            self.name = "Anonymous"

m1 = Member("", 20, 3)
m2 = Member("Bob", 25)

# Print #1
print(1, m1.name, m1.score) # Anonymous, 60

# Print #2
print(2, m2.score) # 50

# Print #3
try:
    print(3, m1.multiplier)
except AttributeError:
    print(3, "AttributeError") # AttributeError because multiplier is never bound to the object dict.

1 Anonymous 60
2 50
3 2


In [56]:
print("class attr:", Member.multiplier)   # does the class itself have it?
print("instance dict:", m1.__dict__)      # is it in the instance's own storage?
print("m1.multiplier:", m1.multiplier)    # where does this actually resolve from?

class attr: 2
instance dict: {'name': 'Anonymous', 'age': 20, 'score': 60}
m1.multiplier: 2


In [59]:
@dataclass(frozen=True)
class Transaction:
    amount: float
    tax_rate: float = 0.05
    total: float = 0.0

    def __post_init__(self):
        # We want to automatically compute 'total'
        #self.total = self.amount * (1 + self.tax_rate)
        object.__setattr__(self, 'total', self.amount*(1+self.tax_rate))

#What happens when you attempt to instantiate this class (e.g., t = Transaction(100.0))? 
# Does it succeed, or does it fail with an exception? If it fails, name the exact exception.
t = Transaction(100.0)
# How do you bypass this limitation inside __post_init__ to successfully set a computed field on an immutable (frozen) dataclass? Explain the exact Python mechanism used to write the value.


## Topic 7: Class Builders Side-by-Side (Comparative Deep-Dive)
1. What It Is
- While all class builders in Chapter 5 automate the generation of record-like data containers, they expose different APIs for standard record operations like dictionary conversion, field inspection, and copying with modifications.
2. The Mechanism: Namespace Preservation
- You might have noticed a stark division in how these builders are designed:
    - Tuple Subclasses (collections.namedtuple, typing.NamedTuple): Provide helper methods directly on the instance prefixed with a single underscore (e.g., x._asdict(), x._replace()).
    - Dataclasses (@dataclass): Provide utility operations as module-level functions in the dataclasses module (e.g., dataclasses.asdict(x), dataclasses.replace(x)).

Why the difference? Named tuples inherit from the built-in tuple. Tuples have highly restricted namespaces (offering only .count() and .index()). To provide record helpers without risking a naming collision with user-defined field names, the designers prefixed these methods with an underscore. Because the underscore means "internal/non-public" by convention, it guarantees that no public tuple field will ever clash with them.

In contrast, `@dataclass` modifies a user-defined class inheriting from object. The class might have complex custom methods, properties, and attributes. To keep the class's public namespace completely pristine and free of any library-specific baggage, the designers chose module-level functions. The decorator does not pollute the instance namespace with a single non-essential method.

3. How does `replace()` work?
- NamedTuple: `instance._replace(**kwargs)`
Because a named tuple is a subclass of the immutable built-in tuple, you cannot reassign any of its fields.
    - The Mechanism: `_replace` is an instance method. It reads the values of the existing instance, overrides them with the key-value pairs you pass as keyword arguments, and passes the entire unified set of values into the class constructor to build and return a brand-new instance of the named tuple.

- Dataclass: `dataclasses.replace(instance, **kwargs)`
Because `@dataclass` works on standard user classes inheriting from object, the class namespace is highly dynamic.
    - The Mechanism: To keep your class's public namespace completely clean, replace is a module-level function.
Under the hood: It copies the instance by invoking the class's autogenerated `__init__` constructor. It retrieves the current values of all fields with `init=True`, overrides them with the keyword arguments you passed, and calls the constructor to yield a brand-new instance.

- Both `_replace()` and `replace()` create a shallow copy and use the same reference of any unmodified mutable fields so any mutation of such mutable fields will be reflected in the new instances.

4. Gotchas & Trade-offs (The Non-Obvious)
- Recursive Deep-Copy vs. Shallow Reference Copy: Calling `namedtuple._asdict()` constructs a dictionary mapping fields to values, but it performs a **shallow** copy of those values. In contrast, `dataclasses.asdict()` recursively processes and converts any nested dataclasses, lists, dicts, and tuples into primitive dict representations. This recursive traversal makes `asdict()` highly convenient for JSON serialization, but it makes it extremely CPU- and memory-intensive compared to the lightweight, flat copy performed by `_asdict()`.
- Mutability of replace: `dataclasses.replace()` can copy both mutable and frozen dataclasses. However, under the hood, it works by invoking the class's autogenerated `__init__` constructor using the modified values. If you set `init=False` on a field, `replace()` cannot set it directly via parameter arguments, which can lead to initialization errors or unexpected defaults.

In [72]:
from typing import NamedTuple
from dataclasses import dataclass, asdict, replace

class ItemNT(NamedTuple):
    name: str
    categories: list

@dataclass
class ItemDC:
    name: str
    categories: list

# Setup: Both instances share the exact same list in memory
shared_list = ['Organic', 'Produce']
item_nt = ItemNT('Apple', shared_list)
item_dc = ItemDC('Apple', shared_list)

# Action 1: Convert both to dictionaries
dict_nt = item_nt._asdict()
dict_dc = asdict(item_dc)

# Action 2: Mutate the original shared list
shared_list.append('Fruit')

print(item_nt.categories, item_nt.categories)

# Print #1
print(1, dict_nt['categories'])

# Print #2
print(2, dict_dc['categories'])

# Print #3
# We create copies using replace utilities, modifying the name
new_nt = item_nt._replace(name='Gala Apple')
new_dc = replace(item_dc, name='Gala Apple')

print(3, new_nt.categories is new_dc.categories)
shared_list.append('FOO')
print(4, new_nt.categories)
print(5, new_dc.categories)


['Organic', 'Produce', 'Fruit'] ['Organic', 'Produce', 'Fruit']
1 ['Organic', 'Produce', 'Fruit']
2 ['Organic', 'Produce']
3 True
4 ['Organic', 'Produce', 'Fruit', 'FOO']
5 ['Organic', 'Produce', 'Fruit', 'FOO']


## Topic 8: Pattern Matching Class Instances
1. What It Is
- Introduced in Python 3.10, PEP 634 structural pattern matching allows you to match class instances using Class Patterns. Class patterns inspect the type of an object and deconstruct its attributes, matching either by attribute name (Keyword Class Patterns) or by position (Positional Class Patterns).

2. The Mechanism: __match_args__
- To understand how positional patterns work under the hood, we must first look at the special class-level attribute `__match_args__`.
- Keyword Matching requires no setup: When you write a keyword match:`case Point(x=10, y=val)`:
Python simply executes standard attribute lookups on the subject instance (checking `subject.x` and `subject.y`). This works on any standard Python class with public instance attributes out of the box.
- Positional Matching requires a Map: When you write a positional match:`case Point(10, val)`:
Python has no native way of knowing which attribute corresponds to the first positional argument, and which corresponds to the second.
- The Protocol: To bridge this gap, Python looks for a class-level attribute named `__match_args__` on the class of the subject. `__match_args__` must be a tuple of **strings** naming the instance attributes in the order they should map to positional subpatterns:
```
    class Point:
        __match_args__ = ('x', 'y')  # Maps 1st position to 'x', 2nd to 'y'
```
- Auto-generation by Builders: The three class builders we've studied (`namedtuple`, `typing.NamedTuple`, and `@dataclass`) automatically generate the `__match_args__` class attribute for you, mapping to their declared field names in order. For standard classes built by hand, you **must write this tuple explicitly**.
3. Gotchas & Edge Cases
- The Nine "Blessed" Built-ins: There is a major exception to the rule. Python designates nine built-in types as "blessed": `bytes`, `dict`, `float`, `frozenset`, `int`, `list`, `set`, `str`, and `tuple`. If you match against these classes (e.g., `case float(val):`), the argument inside the parentheses does **not** represent an attribute lookup. Instead, it matches and binds the **entire subject** instance to the variable.
- Combining Styles: You can freely combine positional and keyword arguments in a pattern (e.g., `case Point(10, y=val):`). However, just like standard function calls, all positional patterns must precede any keyword patterns in the case clause.

In [68]:
class Character:
    __match_args__ = ('name', 'role')
    def __init__(self, name, role, level=1):
        self.name = name
        self.role = role
        self.level = level

# A standard class without __match_args__
class SimpleSkill:
    def __init__(self, name, damage):
        self.name = name
        self.damage = damage

def process_subject(subject):
    match subject:
        # Match pattern A
        case Character("Hero", job, level=10):
            return f"A: Max Hero with job {job}"
        
        # Match pattern B
        case Character("Hero", job):
            return f"B: Low-level Hero with job {job}"

         # Match pattern D
        case SimpleSkill(name="Slash", damage=dmg):
            return f"D: Keyword Slash with {dmg} damage"
        
        # Match pattern C
        case SimpleSkill("Slash", dmg):
            return f"C: Positional Slash with {dmg} damage"
        
       
        # Match pattern E
        case float(val):
            return f"E: Float value is {val}"
            
        case _:
            return "No Match"

# Subjects to process
c1 = Character("Hero", "Warrior", 1)
c2 = SimpleSkill("Slash", 50)
c3 = 42.5

print(1, process_subject(c1)) #"B: Low-level Hero with job Warrior"
print(2, process_subject(c2)) #"D: Keyword Slash with 50 damage"
print(3, process_subject(c3)) #"E: Float value is 42.5"

1 B: Low-level Hero with job Warrior
2 D: Keyword Slash with 50 damage
3 E: Float value is 42.5


In [69]:
70286 - (54165+46480)*.02 - 67486

787.1000000000058

## Topic 9: Data Class as a Code Smell
1. What It Is
- Whether built manually or generated using a standard library builder (like `@dataclass` or `NamedTuple`), a class that contains fields, basic getters/setters, and absolutely no business logic is known as a Data Class.
- In software engineering, Fowler and Beck categorize this design pattern as a Code Smell—a surface-level indicator of a potentially deeper architectural problem in your system. As they famously write: "Data classes are like children. They are okay as a starting point, but to participate as a grownup object, they need to take some responsibility."
2. The Mechanism: Encapsulation vs. Dumb Holders
- The fundamental premise of Object-Oriented Programming (OOP) is to keep data and the behavior that manipulates that data together in the same code unit: the class. 
- When you design your program around "dumb" data holders:
    - The Smell: The class itself has no behavior.
    - The Consequence: The functions and classes using those records must manipulate the records' internal attributes directly.
- The Architecture Breakdown: Because the data class does not protect or process its own state, the business rules (validations, computations) inevitably become scattered, duplicated, and hardcoded throughout different parts of your system. This violates encapsulation, tightly couples external code to your class structure, and introduces severe maintenance overhead.
3. Gotchas & The Two Valid Exceptions
Luciano Ramalho highlights that data classes are not inherently evil; they are highly useful in two specific, well-defined scenarios:
- Data Class as Scaffolding: A temporary, initial implementation to kickstart a module. As your system evolves, you must proactively identify where external functions are manipulating this data and refactor by moving those responsibilities (methods) directly back into the class.
- Data Class as Intermediate Representation (Data Transfer Objects / DTOs): Holding records that have just been parsed from a database, file, or a JSON API, or are about to be exported across a system boundary.
The Strict Rule: In this scenario, the records should be treated as strictly immutable. You should not mutate their attributes on the fly; if changes are required during export/import, you should write specialized custom builder methods rather than modifying the naked records